In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

In [ ]:
# Pasta de trabalho local (repo root, já commitada — ver ingest_prf_sc.py na raiz)
DATA_DIR = Path("../data_prf_sc")
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed"

# Anos a baixar (agrupados por ocorrência). Ajuste conforme necessário —
# quanto mais anos, mais volume para o treino, mas atenção a mudanças de
# metodologia de coleta da PRF ao longo do tempo.
GDRIVE_FILE_IDS: dict[int, str] = {
    # 2026: "1A3IirNm0AzRaSosA1IS94DOVmvKsn0Ol",
    2025: "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR",
    2024: "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO",
    2023: "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq",
    2022: "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-",
    2021: "12xH8LX9aN2gObR766YN3cMcuycwyCJDz",
    2020: "1esu6IiH5TVTxFoedv6DBGDd01Gvi8785",
    # 2019: "1pN3fn2wY34GH6cY-gKfbxRJJBFE0lb_l",
    # 2018: "1cM4IgGMIiR-u4gBIH5IEe3DcvBvUzedi",
    # 2017: "1HPLWt5f_l4RIX3tKjI4tUXyZOev52W0N",
}

UF_ALVO = "SC"

# Aponte aqui para a planilha do SNV baixada manualmente do DNIT/VGEO
SNV_LOCAL_PATH = DATA_DIR / "snv_base.csv"  # .xls também funciona, ver load_snv()

FINAL_OUTPUT_PATH = OUT_DIR / "sinistros_sc_todas_brs.csv"

In [ ]:
def download_prf_year(year: int, file_id: str, dest_dir: Path) -> Path:
    """Baixa o CSV de um ano via gdown (lida com a tela de confirmação do Drive)."""
    import gdown

    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"acidentes{year}.csv"
    if dest.exists():
        print(f"[{year}] já baixado, pulando.")
        return dest

    url = f"https://drive.google.com/uc?id={file_id}"
    print(f"[{year}] baixando...")
    gdown.download(url, str(dest), quiet=False)
    return dest

In [ ]:
import re

def read_prf_csv(path: Path) -> pd.DataFrame:
    """Le CSV da PRF. Arquivo do Drive vem como .zip (mesmo com extensao .csv
    no nome), às vezes contendo o csv dentro de uma subpasta (ex.: datatran2026/
    datatran2026.csv) — por isso não dá pra usar compression="zip" direto, que
    só aceita zip com um único membro. Extraímos o .csv manualmente."""
    import zipfile

    with open(path, "rb") as f:
        is_zip = f.read(2) == b"PK"

    if is_zip:
        with zipfile.ZipFile(path) as zf:
            csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            if not csv_names:
                raise ValueError(f"Nenhum .csv encontrado dentro do zip {path}")
            with zf.open(csv_names[0]) as fh:
                return pd.read_csv(
                    fh, sep=";", encoding="latin1", decimal=",", low_memory=False
                )

    return pd.read_csv(
        path,
        sep=";",
        encoding="latin1",
        decimal=",",
        low_memory=False,
    )


def normalize_prf_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de coluna: minusculas, sem espaco/acento nas bordas."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
        .str.replace(" ", "_", regex=False)
    )
    return df


def _to_nullable_int(series: pd.Series) -> pd.Series:
    """Converte identificadores para inteiro com suporte a nulos."""
    normalized = (
        series.astype(str)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        .str.replace(r"\.0+$", "", regex=True)
        .str.extract(r"(-?\d+)", expand=False)
    )
    return pd.to_numeric(normalized, errors="coerce").astype("Int64")


def _clean_decimal_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"[^0-9,\.\-+]", "", text)
    if "," in text and "." in text:
        # Usa como decimal o último separador; os demais viram milhares.
        comma_pos = text.rfind(",")
        dot_pos = text.rfind(".")
        decimal_sep = "," if comma_pos > dot_pos else "."
        thousands_sep = "." if decimal_sep == "," else ","
        text = text.replace(thousands_sep, "")
        text = text.replace(decimal_sep, ".")
    elif "," in text:
        text = text.replace(",", ".")
    return text


def _coerce_coord_value(value: object, limit: float) -> float | None:
    if pd.isna(value):
        return None
    
    text = _clean_decimal_text(value)
    if text in {"", "+", "-", ".", "+.", "-."}:
        return None

    parsed = pd.to_numeric(text, errors="coerce")
    if pd.isna(parsed):
        return None

    number = float(parsed)
    if abs(number) <= limit:
        return number

    digits = re.sub(r"\D", "", text)
    if not digits:
        return None
    sign = -1 if text.startswith("-") else 1
    scaled = float(int(digits)) * sign
    while abs(scaled) > limit and scaled != 0:
        scaled /= 10.0
    return scaled if abs(scaled) <= limit else None


def normalize_prf_values(df: pd.DataFrame) -> pd.DataFrame:
    """Padroniza tipos sensíveis (id, km, br, latitude, longitude)."""
    df = df.copy()

    for id_col in ("id", "id_acidente"):
        if id_col in df.columns:
            df[id_col] = _to_nullable_int(df[id_col])

    if "km" in df.columns:
        df["km"] = pd.to_numeric(
            df["km"].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

    if "br" in df.columns:
        df["br"] = pd.to_numeric(
            df["br"].astype(str).str.extract(r"(\d+)", expand=False),
            errors="coerce",
        ).astype("Int64")

    if "latitude" in df.columns:
        df["latitude"] = df["latitude"].map(lambda value: _coerce_coord_value(value, 90.0))
    if "longitude" in df.columns:
        df["longitude"] = df["longitude"].map(lambda value: _coerce_coord_value(value, 180.0))

    if "br" in df.columns:
        df["fora_br"] = df["br"].fillna(-1).eq(0)
        df["relevancia_rodovia"] = df["fora_br"].map(
            {True: "baixa (fora de BR)", False: "alta (em BR)"}
        )

    return df

In [ ]:
def filter_sc_brs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # coluna de km costuma vir como string com vírgula decimal
    if "km" in df.columns:
        df["km"] = pd.to_numeric(
            df["km"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
        )
    if "br" in df.columns:
        df["br"] = pd.to_numeric(df["br"], errors="coerce")

    mask = pd.Series(True, index=df.index)
    if "uf" in df.columns:
        mask &= df["uf"].astype(str).str.upper().str.strip() == UF_ALVO
    return df[mask].reset_index(drop=True)

In [ ]:
def load_snv(path: Path) -> pd.DataFrame:
    """
    Carrega a planilha do SNV. Ajuste os nomes de coluna abaixo conforme a
    versão baixada (eles variam um pouco entre releases do DNIT).
    Colunas esperadas após normalização: br, uf, km_inicial, km_final,
    superficie, jurisdicao (algumas dessas podem não existir — trate como opcional).
    """
    if not path.exists():
        print(
            f"[aviso] {path} não encontrado — pulando o merge com o SNV. "
            "Baixe a planilha do DNIT/VGEO e aponte SNV_LOCAL_PATH para rodar essa etapa."
        )
        return pd.DataFrame()

    if path.suffix.lower() in {".xls", ".xlsx"}:
        snv = pd.read_excel(path)
    else:
        snv = pd.read_csv(path, sep=None, engine="python")

    snv = normalize_prf_columns(snv)  # mesma limpeza de nomes serve aqui
    return snv


def merge_with_snv(acidentes: pd.DataFrame, snv: pd.DataFrame) -> pd.DataFrame:
    """
    Casa cada acidente ao trecho do SNV cujo intervalo [km_inicial, km_final]
    contém o km do acidente, para o mesmo BR/UF.
    """
    if snv.empty or "km" not in acidentes.columns:
        return acidentes

    required = {"br", "uf", "km_inicial", "km_final"}
    missing = required - set(snv.columns)
    if missing:
        print(f"[aviso] SNV sem as colunas {missing} — confira os nomes reais e ajuste load_snv(). Pulando merge.")
        return acidentes

    snv = snv.copy()
    snv["br"] = pd.to_numeric(snv["br"], errors="coerce")
    snv["km_inicial"] = pd.to_numeric(snv["km_inicial"], errors="coerce")
    snv["km_final"] = pd.to_numeric(snv["km_final"], errors="coerce")

    merged_rows = []
    for (uf, br), grp_acid in acidentes.groupby(["uf", "br"]):
        grp_snv = snv[(snv["uf"].astype(str).str.upper() == str(uf).upper()) & (snv["br"] == br)]
        if grp_snv.empty:
            merged_rows.append(grp_acid)
            continue

        grp_snv = grp_snv.sort_values("km_inicial")
        acids = grp_acid.copy()
        acids["_snv_idx"] = acids["km"].apply(
            lambda k: _find_segment(k, grp_snv) if pd.notnull(k) else None
        )
        acids = acids.merge(
            grp_snv.drop(columns=["br", "uf"]).add_prefix("snv_"),
            left_on="_snv_idx",
            right_index=True,
            how="left",
        ).drop(columns=["_snv_idx"])
        merged_rows.append(acids)

    return pd.concat(merged_rows, ignore_index=True) if merged_rows else acidentes


def _find_segment(km: float, snv_group: pd.DataFrame):
    match = snv_group[(snv_group["km_inicial"] <= km) & (km <= snv_group["km_final"])]
    return match.index[0] if not match.empty else None

## Arquivo Main()

cria as pastas e faz o download das planilhas faltantes
dataset final pronto pra EDA, sinistros de SC em todas as BRs, ano a ano.

In [ ]:
def main() -> None:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    frames = []
    for year, file_id in sorted(GDRIVE_FILE_IDS.items()):
        try:
            path = download_prf_year(year, file_id, RAW_DIR)
            df = read_prf_csv(path)
            df = normalize_prf_columns(df)
            df["ano_arquivo"] = year
            df_sc = filter_sc_brs(df)
            print(f"[{year}] {len(df)} registros nacionais -> {len(df_sc)} em SC/todas as BRs")
            frames.append(df_sc)
        except Exception as e:  # noqa: BLE001
            print(f"[{year}] ERRO: {e}", file=sys.stderr)

    if not frames:
        print("Nenhum ano foi processado com sucesso. Abortando.")
        return

    acidentes = pd.concat(frames, ignore_index=True)
    interim_path = OUT_DIR / "acidentes_sc_todas_brs_bruto.csv"
    acidentes.to_csv(interim_path, index=False, decimal=",")
    print(f"\nBase filtrada (sem merge SNV) salva em: {interim_path} ({len(acidentes)} linhas)")

    br_summary = (
        acidentes.groupby("br", dropna=False)
        .size()
        .rename("quantidade_acidentes")
        .reset_index()
        .sort_values("br", na_position="last")
    )
    br_summary.to_csv(OUT_DIR / "brs_sc_resumo.csv", index=False)
    print(f"Resumo das BRs em SC salvo em: {OUT_DIR / 'brs_sc_resumo.csv'}")

    snv = load_snv(SNV_LOCAL_PATH)
    final = merge_with_snv(acidentes, snv)

    final.to_csv(FINAL_OUTPUT_PATH, index=False, decimal=",")
    print(f"\nDataset final pronto para EDA: {FINAL_OUTPUT_PATH} ({len(final)} linhas, {final.shape[1]} colunas)")

    # resumo rápido, útil como primeiro check da EDA
    print("\nResumo rápido:")
    print(final.dtypes.value_counts())
    print("\nValores nulos por coluna (top 10):")
    print(final.isnull().sum().sort_values(ascending=False).head(10))


main()

## Gráficos exploratórios

Painel rápido sobre a base já filtrada (SC / todas as BRs), pra primeira leitura antes da EDA completa.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Paleta fixa (evita cor "no chute" e evita rainbow)
COLOR_BLUE = "#2a78d6"
COLOR_ORANGE = "#eb6834"
COLOR_GOOD = "#0ca30c"
COLOR_WARNING = "#fab219"
COLOR_SERIOUS = "#ec835a"
COLOR_CRITICAL = "#d03b3b"
GRID_COLOR = "#e1e0d9"
DIAS_ORDEM = [
    "segunda-feira", "terça-feira", "quarta-feira", "quinta-feira",
    "sexta-feira", "sábado", "domingo",
]


def _style_ax(ax, title: str) -> None:
    ax.set_title(title, fontsize=12, loc="left", pad=10)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")


def _finish_pair(fig) -> None:
    fig.tight_layout(pad=2.2, w_pad=3.0)
    plt.show()


def plot_overview(df: pd.DataFrame) -> None:
    if df.empty:
        print("Base vazia — nada para plotar.")
        return

    brs = sorted(pd.to_numeric(df["br"], errors="coerce").dropna().unique())
    br_colors = {br: plt.cm.tab20(index % 20) for index, br in enumerate(brs)}

    # Figura 1: volume geral
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    ax = axes[0]
    contagem_br = df["br"].value_counts().reindex(brs).fillna(0)
    ax.bar(
        [str(int(br)) for br in contagem_br.index],
        contagem_br.values,
        color=[br_colors[br] for br in contagem_br.index],
        width=0.6,
        zorder=3,
    )
    ax.set_xlabel("Rodovia (BR)")
    ax.set_ylabel("Quantidade de acidentes")
    _style_ax(ax, "Acidentes por BR em SC")

    ax = axes[1]
    if "ano_arquivo" in df.columns:
        por_ano = df.groupby("ano_arquivo").size().sort_index()
        ax.plot(por_ano.index, por_ano.values, color=COLOR_BLUE, linewidth=2, marker="o", markersize=6, zorder=3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.set_xlabel("Ano")
        ax.set_ylabel("Quantidade de acidentes")
    else:
        ax.text(0.5, 0.5, "sem coluna ano_arquivo", ha="center", va="center", transform=ax.transAxes)
    _style_ax(ax, "Acidentes por ano")
    _finish_pair(fig)

    # Figura 2: principais características dos acidentes
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]
    if "causa_acidente" in df.columns:
        top_causas = df["causa_acidente"].value_counts().head(10).sort_values()
        ax.barh(top_causas.index, top_causas.values, color=COLOR_BLUE, zorder=3)
        ax.tick_params(axis="y", labelsize=9)
        ax.set_xlabel("Quantidade de acidentes")
    else:
        ax.text(0.5, 0.5, "sem coluna causa_acidente", ha="center", va="center", transform=ax.transAxes)
    _style_ax(ax, "Top 10 causas de acidente")

    ax = axes[1]
    if "dia_semana" in df.columns:
        dia_norm = df["dia_semana"].astype(str).str.lower().str.strip()
        presentes = [d for d in DIAS_ORDEM if d in dia_norm.unique()]
        ordem = presentes if presentes else sorted(dia_norm.unique())
        contagem_dia = dia_norm.value_counts().reindex(ordem).fillna(0)
        ax.bar(range(len(contagem_dia)), contagem_dia.values, color=COLOR_BLUE, zorder=3)
        ax.set_xticks(range(len(contagem_dia)))
        ax.set_xticklabels([d[:3] for d in contagem_dia.index], fontsize=9)
        ax.set_xlabel("Dia da semana")
        ax.set_ylabel("Quantidade de acidentes")
    else:
        ax.text(0.5, 0.5, "sem coluna dia_semana", ha="center", va="center", transform=ax.transAxes)
    _style_ax(ax, "Acidentes por dia da semana")
    _finish_pair(fig)

    # Figura 3: severidade e localização dos acidentes
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]
    severidade_cols = ["ilesos", "feridos_leves", "feridos_graves", "mortos"]
    disponiveis = [c for c in severidade_cols if c in df.columns]
    if disponiveis and "br" in df.columns:
        sev_por_br = df.groupby("br")[disponiveis].sum().reindex(brs).fillna(0)
        cores = {"ilesos": COLOR_GOOD, "feridos_leves": COLOR_WARNING,
                 "feridos_graves": COLOR_SERIOUS, "mortos": COLOR_CRITICAL}
        x = range(len(sev_por_br))
        bottom = [0] * len(sev_por_br)
        for col in disponiveis:
            ax.bar(x, sev_por_br[col], bottom=bottom, label=col.replace("_", " "),
                   color=cores[col], width=0.6, zorder=3)
            bottom = [b + v for b, v in zip(bottom, sev_por_br[col])]
        ax.set_xticks(list(x))
        ax.set_xticklabels([str(int(br)) for br in sev_por_br.index], fontsize=9)
        ax.set_xlabel("Rodovia (BR)")
        ax.set_ylabel("Quantidade de pessoas")
        ax.legend(fontsize=9, frameon=False, loc="upper right")
    else:
        ax.text(0.5, 0.5, "colunas de severidade ausentes", ha="center", va="center", transform=ax.transAxes)
    _style_ax(ax, "Severidade por BR (pessoas)")

    ax = axes[1]
    if "km" in df.columns and "br" in df.columns:
        for br in brs:
            valores = df.loc[df["br"] == br, "km"].dropna()
            if not valores.empty:
                ax.hist(valores, bins=30, color=br_colors[br], alpha=0.55, label=f"BR-{int(br)}", zorder=3)
        ax.set_xlabel("Quilômetro da rodovia")
        ax.set_ylabel("Quantidade de acidentes")
        ax.legend(fontsize=9, frameon=False, ncol=2)
    else:
        ax.text(0.5, 0.5, "sem colunas km/br", ha="center", va="center", transform=ax.transAxes)
    _style_ax(ax, "Distribuição de km por BR")
    _finish_pair(fig)


if FINAL_OUTPUT_PATH.exists():
    _df_plot = pd.read_csv(FINAL_OUTPUT_PATH)
    plot_overview(_df_plot)
else:
    print(f"{FINAL_OUTPUT_PATH} não existe ainda — rode main() primeiro.")